# Plans against shop drawings: end-to-end demonstration

This notebook explores one project and runs the whole pipeline on it, from the PDFs to the findings
report. Everything runs locally; no document leaves the machine.

**Project layout.** The structural plans are the PDFs at the top of the project folder; the shop
drawings are the PDFs anywhere under its `DA/` folder.

**Pipeline.** Read each page's text with its position (OCR for scanned pages) → group the words into
notes and free-form elements (kind, grid cell, level, bars, characteristics) → match plan and shop
elements (grid cell, level, kind, content; Hungarian assignment) → compare the reinforcement with
unit normalisation → one PDF report per project, organised by plan sheet.

Set `PROJECT` below (or the `L2C_PROJECT` environment variable) to any project folder. Outputs go to
`deliverables/notebook_run/` (git-ignored). Setup: see the README (`uv pip install -r requirements.txt`,
`uv pip install -e .`).

In [ ]:
import json
import os
import re
import sys
import time
from collections import Counter
from pathlib import Path

import pymupdf
from IPython.display import Image, Markdown, display

# the repository root: the folder holding backend/, from wherever the notebook was opened
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend" / "l2c").is_dir())
try:
    import l2c  # noqa: F401  (installed with `uv pip install -e .`)
except ImportError:
    sys.path.insert(0, str(REPO / "backend"))

# the project to run: plans at its top, shop drawings under DA/ (or set L2C_PROJECT)
PROJECT = Path(os.environ.get("L2C_PROJECT", REPO / "data" / "raw" / "CLP"))
OUT = Path(os.environ.get("L2C_OUT", REPO / "deliverables" / "notebook_run"))
if not PROJECT.is_dir():
    raise FileNotFoundError(f"No project folder at {PROJECT}: set PROJECT above or the L2C_PROJECT variable")


def table(header, rows):
    """A small Markdown table."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


def show_page(pdf, index=0, width=900, dpi=36):
    with pymupdf.open(pdf) as doc:
        display(Image(data=doc[index].get_pixmap(dpi=dpi).tobytes("png"), width=width))


print("project:", PROJECT)

## 1. Data exploration

What the project holds: plan and shop files, pages, and how many pages have no usable text layer
(scanned or mostly raster), which must go through OCR.

In [ ]:
from l2c.pipeline.__main__ import discover
from l2c.pipeline.stream import _needs_ocr

files = discover(PROJECT)
rows = []
for pdf, rel, source in files:
    with pymupdf.open(pdf) as doc:
        scanned = sum(_needs_ocr(p) for p in doc)
        words = sum(len(p.get_text("words")) for p in doc)
        size = doc[0].rect
        rows.append((source, rel, len(doc), scanned, words, f"{size.width / 72:.0f} x {size.height / 72:.0f} in"))
table(["source", "file", "pages", "pages needing OCR", "words in text layer", "page size"], rows)
by = Counter(r[0] for r in rows)
print(f"{by['plan']} plan file(s), {by['shop']} shop files, {sum(r[2] for r in rows)} pages, {sum(r[3] for r in rows)} need OCR")

A plan sheet and a shop drawing, at low resolution. Reinforcement is written as notes next to
the members (bar count, size, spacing), and the grid (lettered rows, numbered columns) locates them.

In [ ]:
plan_pdf = next(pdf for pdf, _, s in files if s == "plan")
shop_pdf = next((pdf for pdf, _, s in files if s == "shop"), None)
show_page(plan_pdf)
if shop_pdf:
    show_page(shop_pdf)

The reinforcement notation in the text layer: the most frequent words that look like bar notes
(a size such as `20M`, a spacing `@`).

In [ ]:
NOTE = re.compile(r"\d+M\b|@")
for label, pdf in (("plan", plan_pdf), ("shop", shop_pdf)):
    if pdf is None:
        continue
    with pymupdf.open(pdf) as doc:
        words = [w[4] for p in doc for w in p.get_text("words")]
    print(label, Counter(w for w in words if NOTE.search(w)).most_common(12))

## 2. Run the pipeline end to end

`run` is the same streaming run the web app uses: plans first, each file's elements written as soon
as it is read, partial findings while the rest loads, then the final pass and the report. The caches
are off here, so every file is read. No model questions (`llm="none"`).

In [ ]:
from l2c.pipeline.stream import run

t0 = time.time()
findings = run(PROJECT, OUT, llm="none", ocr_cache=None, file_cache=None, partial_every=5.0)
print(f"\nend to end: {time.time() - t0:.0f} s")

## 3. The extracted elements (free-form JSON)

One file per PDF in `elements/`. Each element has default fields (id, source, file, sheet, page,
x, y in PDF points from the page's top-left corner, kind, name) and open lists: locations (grid
cell, level), bars (count, size, spacing, mark, as read), characteristics and descriptions. A kind
the pipeline has never seen is kept as it is written.

In [ ]:
elements = []
for f in sorted((OUT / "elements").glob("elements.*.json")):
    elements += json.loads(f.read_text(encoding="utf-8"))["elements"]
per_file = Counter((e["source"], Path(e["file"]).name) for e in elements)
table(["source", "file", "elements"], [(s, n, c) for (s, n), c in sorted(per_file.items())])
print("kinds:", Counter(e.get("kind") or "unknown" for e in elements).most_common())
print("with bars:", sum(1 for e in elements if e.get("bars")), "of", len(elements))

In [ ]:
def brief(e):
    keep = ("id", "source", "file", "sheet", "page", "x", "y", "kind", "name", "locations", "bars", "characteristics")
    return json.dumps({k: e.get(k) for k in keep}, ensure_ascii=False, indent=1)[:1500]


for source in ("plan", "shop"):
    example = next((e for e in elements if e["source"] == source and e.get("bars")), None)
    print(brief(example) if example else f"no {source} element with bars")

## 4. Matching and comparison

Every plan element ends in one record: compliant, non-compliant, to verify (a possible difference,
or a pairing too weak to call), missing from the shop drawings, or added in them. Shop drawings do
not cover every plan element, so "missing" is information, not an error.

In [ ]:
print(findings["counts"])
by_sheet = {}
for r in findings["entities"]:
    if not r["members"]["plan"]:
        continue
    sheet = r["members"]["plan"][0].get("sheet") or f"page {r['members']['plan'][0]['page']}"
    by_sheet.setdefault(sheet, Counter())[r["status"]] += 1
table(
    ["plan sheet", "compliant", "non-compliant", "to verify", "missing from shop"],
    [(s, c["conforms"], c["differs"], c["uncertain"], c["not_in_shop"]) for s, c in sorted(by_sheet.items())],
)

The non-conformities, with where to find them on both documents (page and X, Y in PDF points).

In [ ]:
def where(m):
    return f"{Path(m['file']).name} p{m['page']} ({m['x']:.0f}, {m['y']:.0f})"


rows = []
for r in findings["entities"]:
    if r["status"] != "differs":
        continue
    for f in r["flags"]:
        shop = f.get("shop_ref") or (r["members"]["shop"] or [None])[0]
        rows.append((f"{r['kind'] or 'element'} {r['location']['grid']}", r["level"], f["property"], f["plan"], f["shop"], where(r["members"]["plan"][0]), where(shop) if shop else "-"))
table(["element", "level", "property", "plan", "shop", "plan position", "shop position"], rows)
for g in findings.get("systematic", []):
    print(f"systematic {g['group']}: {g['property']}, {g['relation']}, at {g['count']} places")

## 5. The report

One PDF for the project: a summary, a table per plan sheet, the discrepancies with their position on
both documents, the elements missing from or added in the shop drawings, and the method and its
limits. Its first page:

In [ ]:
report = OUT / "findings.pdf"
print(report, f"{report.stat().st_size // 1024} KB")
show_page(report, width=800, dpi=60)

## Known limits

- Scanned projects are bound by OCR time: about 8 to 10 s per large sheet on a 4 GB laptop GPU. The
  web app streams results meanwhile, and a folder cache skips files already read.
- On scanned projects, grid labels read by OCR do not always match the plan's grid names, which
  leaves many elements unpaired.
- A difference counts as a non-conformity only when the rest of the bar group agrees; otherwise it is
  listed to verify. The same difference at several places is reported once, as systematic.
- Planted-change test (`python -m l2c.pipeline.evaluate <project>`): on the development project about
  80% of single-number changes are found.